# **Filtrado Basado en Contenido con Embeddings BERT**

En este notebook, implementamos una versión avanzada del filtrado basado en contenido que incorpora embeddings contextuales (BERT) para mejorar la comprensión semántica de las características textuales y generar recomendaciones más precisas.

### **Importación de componentes**

In [3]:
import sys
import os

# Configuramos el path del proyecto
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.append(project_root)

# Lista para registrar qué componentes se importaron exitosamente
importados = []
fallidos = []

# Importamos componentes estándar
try:
    from pipeline.vectorizers import vectorize_text_tfidf, vectorize_numerical
    importados.append("vectorizers")
except ImportError as e:
    print(f"Error al importar vectorizadores: {e}")
    fallidos.append("vectorizers")
    
try:    
    from models.enhanced_content_based import generate_user_vector, recommend_by_content
    importados.append("enhanced_content_based")
except ImportError as e:
    print(f"Error al importar enhanced_content_based: {e}")
    fallidos.append("enhanced_content_based")
    
try:
    from evaluation.metrics import precision_at_k, recall_at_k, ndcg_at_k
    importados.append("metrics")
except ImportError as e:
    print(f"Error al importar métricas: {e}")
    fallidos.append("metrics")

# Comprobamos si podemos importar PyTorch correctamente
torch_ok = False
try:
    import torch
    print(f"PyTorch importado exitosamente. Versión: {torch.__version__}")
    torch_ok = True
except Exception as e:
    print(f"Error al importar PyTorch: {e}")
    fallidos.append("torch")

# Solo intentamos importar los componentes BERT si PyTorch funciona
if torch_ok:
    try:
        # Importamos nuevos componentes para BERT
        from pipeline.embedding_vectorizers import vectorize_text_bert, vectorize_query_bert
        importados.append("embedding_vectorizers")
    except ImportError as e:
        print(f"Error al importar embedding_vectorizers: {e}")
        fallidos.append("embedding_vectorizers")
        
    try:
        from models.neural_content_based import recommend_by_content_bert, recommend_hybrid_content
        importados.append("neural_content_based")
    except ImportError as e:
        print(f"Error al importar neural_content_based: {e}")
        fallidos.append("neural_content_based")

# Mostrar resumen de importaciones
print("\nComponentes importados exitosamente:", ", ".join(importados))
if fallidos:
    print("Componentes con fallos:", ", ".join(fallidos))

TypeError: 'type' object is not subscriptable

### **Instalación de dependencias**

Primero necesitamos instalar las bibliotecas necesarias para trabajar con embeddings BERT.

In [ ]:
# Verificar que PyTorch se importa correctamente
try:
    import torch
    print(f"PyTorch versión: {torch.__version__}")
    print(f"PyTorch instalado en: {torch.__file__}")
    print(f"CUDA disponible: {torch.cuda.is_available()}")
    
    # En Mac con Apple Silicon, verificamos que MPS está disponible
    if hasattr(torch, 'backends') and hasattr(torch.backends, 'mps'):
        print(f"MPS (Metal Performance Shaders) disponible: {torch.backends.mps.is_available()}")
    
    # Probar una operación simple para verificar que funciona
    tensor = torch.rand(3, 3)
    print("Tensor de prueba creado exitosamente:")
    print(tensor)
    
    print("\nPyTorch funciona correctamente.")
except Exception as e:
    print(f"Error al importar PyTorch: {e}")

In [ ]:
# Instalamos las dependencias necesarias con versiones específicas para garantizar compatibilidad
# Primero desinstalamos las versiones actuales que pueden estar causando conflictos
!pip uninstall -y torch torchvision torchaudio transformers sentence-transformers

# Instalamos PyTorch específicamente para Apple Silicon (si estamos en Mac M1/M2/M3)
# Y versiones específicas de transformers y sentence-transformers que son compatibles
%pip install torch==2.0.1 transformers==4.30.2 sentence-transformers==2.2.2

# Verificamos las versiones instaladas
import pkg_resources
for pkg in ['torch', 'transformers', 'sentence-transformers']:
    try:
        version = pkg_resources.get_distribution(pkg).version
        print(f"{pkg}: {version}")
    except pkg_resources.DistributionNotFound:
        print(f"{pkg}: no instalado")

In [ ]:
# Diagnóstico de errores en PyTorch
import os
import sys
import subprocess
import platform

def run_command(cmd):
    """Ejecutar un comando y devolver la salida"""
    try:
        result = subprocess.run(cmd, shell=True, check=True, 
                               stdout=subprocess.PIPE, stderr=subprocess.PIPE, 
                               universal_newlines=True)
        return result.stdout
    except subprocess.CalledProcessError as e:
        print(f"Error ejecutando '{cmd}': {e}")
        print(f"Stderr: {e.stderr}")
        return None

# Información del sistema
print(f"Python versión: {sys.version}")
print(f"Sistema operativo: {platform.system()} {platform.release()}")
print(f"Arquitectura: {platform.machine()}")
print(f"Ubicación de Python: {sys.executable}")

# Si estamos en macOS con Apple Silicon, necesitamos una instalación específica
if platform.system() == "Darwin" and platform.machine() == "arm64":
    print("\n🔍 Detectado macOS con Apple Silicon (M1/M2/M3)")
    print("Recomendación: Usar la versión de PyTorch específica para esta arquitectura")
    
    # Para macOS arm64, usar esta instalación específica
    install_cmd = """
    pip uninstall -y torch torchvision torchaudio
    pip install torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2
    """
    
    print("\n💡 Para solucionar el problema, ejecuta este comando fuera del notebook:")
    print(f"\n{install_cmd}")
    print("Después de la instalación, reinicia el kernel del notebook")
    
    print("\nVerificando ruta de las bibliotecas de PyTorch...")
    for path in sys.path:
        if "torch" in path:
            print(f"- {path}")

print("\nPara continuar con el notebook sin usar BERT, comenta las líneas de importación relacionadas con PyTorch y embedding_vectorizers")
print("y ejecuta solo la parte con TF-IDF.")

## **1. Carga de datos**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuramos el estilo de los gráficos
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('viridis')
plt.rcParams['figure.figsize'] = [10, 6]

# Cargamos los datos preprocesados generados en el cuaderno de preparación de datos
df = pd.read_csv("../data/preprocessed_data.csv")

# Mostramos las primeras filas del DataFrame y la columna de texto
df.head()

In [ ]:
# Examinemos la columna de texto combinado para entender qué información procesará BERT
print(df['text_combined'].iloc[0])
print("\nLongitud promedio de textos:", df['text_combined'].str.len().mean())
print("Número de razas:", len(df))

## **2. Vectorización de características**

Primero generamos los vectores numéricos y TF-IDF (como en el notebook anterior), y luego añadimos los embeddings BERT.

In [ ]:
# Vectorizar características numéricas (igual que antes)
numerical_columns = [
    "grooming_frequency_value",
    "shedding_value",
    "energy_level_value",
    "trainability_value",
    "demeanor_value"
]

X_num, scaler = vectorize_numerical(df, numerical_columns)
print(f"Dimensiones de vectores numéricos: {X_num.shape}")

In [ ]:
# Vectorizar características de texto con TF-IDF (para comparación)
X_tfidf, tfidf_model = vectorize_text_tfidf(df)
print(f"Dimensiones de vectores TF-IDF: {X_tfidf.shape}")

In [ ]:
# Vectorizar características de texto con BERT
# Usamos un modelo pequeño pero eficiente: all-MiniLM-L6-v2
X_bert, bert_model = vectorize_text_bert(df, text_column='text_combined', model_name='all-MiniLM-L6-v2')
print(f"Dimensiones de embeddings BERT: {X_bert.shape}")

## **3. Definición de perfiles de usuario**

Utilizaremos los mismos perfiles de usuario que en el experimento anterior para facilitar la comparación.

In [ ]:
# Creamos perfiles de usuario para pruebas (mismos que en el experimento anterior)
test_profiles = [
    {
        "nombre": "Usuario con familia",
        "grooming_frequency_value": 0.4,  # Poco mantenimiento
        "shedding_value": 0.4,            # Poca caída de pelo
        "energy_level_value": 0.6,        # Energía media
        "trainability_value": 0.8,        # Buena capacidad de entrenamiento
        "demeanor_value": 1.0,            # Muy amigable
        "text_query": "friendly family children"  # Consulta de texto
    },
    {
        "nombre": "Usuario deportista",
        "grooming_frequency_value": 0.6,  # Mantenimiento medio
        "shedding_value": 0.6,            # Caída de pelo media
        "energy_level_value": 1.0,        # Muy energético
        "trainability_value": 1.0,        # Muy entrenable
        "demeanor_value": 0.6,            # Temperamento medio
        "text_query": "energetic active outdoor"  # Consulta de texto
    },
    {
        "nombre": "Usuario en apartamento",
        "grooming_frequency_value": 0.8,  # Alto mantenimiento
        "shedding_value": 0.2,            # Mínima caída de pelo
        "energy_level_value": 0.4,        # Poca energía
        "trainability_value": 0.6,        # Entrenabilidad media
        "demeanor_value": 0.8,            # Bastante amigable
        "text_query": "low energy calm"  # Consulta de texto
    },
    {
        "nombre": "Usuario mayor",
        "grooming_frequency_value": 0.6,  # Mantenimiento medio
        "shedding_value": 0.2,            # Mínima caída de pelo
        "energy_level_value": 0.2,        # Muy poca energía
        "trainability_value": 0.6,        # Entrenabilidad media
        "demeanor_value": 1.0,            # Muy amigable
        "text_query": "calm gentle companion senior low energy"  # Consulta de texto
    },
    {
        "nombre": "Usuario primerizo",
        "grooming_frequency_value": 0.4,  # Poco mantenimiento
        "shedding_value": 0.4,            # Poca caída de pelo
        "energy_level_value": 0.4,        # Poca energía
        "trainability_value": 1.0,        # Muy entrenable
        "demeanor_value": 0.8,            # Bastante amigable
        "text_query": "easy to train beginner friendly low maintenance"  # Consulta de texto
    },
    {
        "nombre": "Usuario con niños pequeños",
        "grooming_frequency_value": 0.6,  # Mantenimiento medio
        "shedding_value": 0.4,            # Poca caída de pelo
        "energy_level_value": 0.6,        # Energía media
        "trainability_value": 0.8,        # Buena capacidad de entrenamiento
        "demeanor_value": 1.0,            # Muy amigable
        "text_query": "patient gentle tolerant with children family"  # Consulta de texto
    },
    {
        "nombre": "Usuario con alergias",
        "grooming_frequency_value": 1.0,  # Muy alto mantenimiento (pelo regular)
        "shedding_value": 0.2,            # Mínima caída de pelo
        "energy_level_value": 0.6,        # Energía media
        "trainability_value": 0.6,        # Entrenabilidad media
        "demeanor_value": 0.8,            # Bastante amigable
        "text_query": "hypoallergenic non shedding low dander"  # Consulta de texto
    },
    {
        "nombre": "Usuario con fines de protección",
        "grooming_frequency_value": 0.4,  # Poco mantenimiento
        "shedding_value": 0.6,            # Caída de pelo media
        "energy_level_value": 0.8,        # Bastante energético
        "trainability_value": 0.8,        # Buena capacidad de entrenamiento
        "demeanor_value": 0.2,            # Reservado/alerta - BAJO para perros guardianes
        "text_query": "protective guard watchful alert territorial"  # Consulta de texto
    }
]

# Creamos una lista de orden de características para los perfiles de usuario
feature_order = [col for col in list(test_profiles[0].keys())[1:] if col != "text_query"]

## **4. Comparación de recomendaciones con diferentes modelos**

Vamos a comparar las recomendaciones generadas por tres enfoques diferentes:
1. Solo características numéricas
2. Características numéricas + TF-IDF
3. Características numéricas + BERT

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Configuración para comparaciones
alpha_values = [0.5]  # Usamos un valor balanceado para comparación
models_to_compare = [
    {"nombre": "Numérico", "alpha": 1.0, "tipo": "tfidf"},
    {"nombre": "Numérico + TF-IDF", "alpha": 0.5, "tipo": "tfidf"},
    {"nombre": "Numérico + BERT", "alpha": 0.5, "tipo": "bert"}
]

# Almacenamos todas las recomendaciones
all_recommendations = {model["nombre"]: {} for model in models_to_compare}

# Para cada perfil, generamos recomendaciones con cada modelo
for profile in test_profiles:
    # Extraemos datos del perfil
    nombre = profile["nombre"]
    text_query = profile["text_query"]
    numeric_preferences = {k: profile[k] for k in feature_order}
    
    # Generamos vector de usuario (componente numérico)
    user_vec = generate_user_vector(numeric_preferences, scaler, feature_order)
    
    # Para cada modelo, generamos recomendaciones
    for model in models_to_compare:
        model_name = model["nombre"]
        alpha = model["alpha"]
        tipo = model["tipo"]
        
        if tipo == "tfidf":
            # Usar modelo TF-IDF
            recs = recommend_by_content(
                user_vec, X_num, df, top_k=10, 
                text_query=text_query if alpha < 1.0 else None, 
                text_vectors=X_tfidf, 
                tfidf_vectorizer=tfidf_model, 
                alpha=alpha
            )
        elif tipo == "bert":
            # Usar modelo BERT
            recs = recommend_by_content_bert(
                user_vec, X_num, df, top_k=10,
                text_query=text_query,
                text_embeddings=X_bert,
                bert_model=bert_model,
                alpha=alpha
            )
        
        # Guardar recomendaciones
        all_recommendations[model_name][nombre] = recs

### **4.1 Análisis de recomendaciones para un perfil específico**

In [ ]:
# Mostramos las recomendaciones para el usuario primerizo con diferentes modelos
usuario_analizado = "Usuario primerizo"

for model_name, recs_by_profile in all_recommendations.items():
    print(f"\nRecomendaciones con modelo {model_name} para {usuario_analizado}:")
    display(recs_by_profile[usuario_analizado][['breed', 'cbf_score', 'group']].head(5))
    
    # Calculamos estadísticas básicas de los scores
    scores = recs_by_profile[usuario_analizado]['cbf_score']
    print(f"Rango de scores: {scores.min():.4f} - {scores.max():.4f}")
    print(f"Media: {scores.mean():.4f}, Desviación estándar: {scores.std():.4f}")

### **4.2 Análisis de solapamiento entre modelos**

In [ ]:
# Analizamos el solapamiento entre los diferentes modelos para cada perfil
modelo_base = "Numérico"
modelos_textuales = ["Numérico + TF-IDF", "Numérico + BERT"]

solapamiento_df = pd.DataFrame(index=[p["nombre"] for p in test_profiles], columns=modelos_textuales)

for profile in test_profiles:
    nombre = profile["nombre"]
    
    # Obtenemos razas recomendadas por el modelo base
    razas_base = set(all_recommendations[modelo_base][nombre]['breed'].tolist())
    
    # Calculamos solapamiento con cada modelo textual
    for modelo in modelos_textuales:
        razas_modelo = set(all_recommendations[modelo][nombre]['breed'].tolist())
        overlap = len(razas_base.intersection(razas_modelo))
        overlap_percent = overlap / 10 * 100  # Top-10 recomendaciones
        solapamiento_df.loc[nombre, modelo] = overlap_percent

# Visualizamos matriz de solapamiento
plt.figure(figsize=(12, 8))
sns.heatmap(
    solapamiento_df,
    annot=True,
    fmt='.1f',
    cmap='YlOrRd',
    vmin=0,
    vmax=100
)
plt.title(f'Solapamiento (%) entre modelo numérico y modelos textuales')
plt.tight_layout()
plt.show()

### **4.3 Comparación directa entre TF-IDF y BERT**

In [ ]:
# Comparamos el solapamiento entre los modelos textuales
solapamiento_textuales = pd.DataFrame(index=[p["nombre"] for p in test_profiles], columns=["Solapamiento TF-IDF vs BERT (%)"])

for profile in test_profiles:
    nombre = profile["nombre"]
    
    razas_tfidf = set(all_recommendations["Numérico + TF-IDF"][nombre]['breed'].tolist())
    razas_bert = set(all_recommendations["Numérico + BERT"][nombre]['breed'].tolist())
    
    overlap = len(razas_tfidf.intersection(razas_bert))
    overlap_percent = overlap / 10 * 100  # Top-10 recomendaciones
    solapamiento_textuales.loc[nombre, "Solapamiento TF-IDF vs BERT (%)"] = overlap_percent

# Visualizar
plt.figure(figsize=(10, 6))
sns.barplot(y=solapamiento_textuales.index, x="Solapamiento TF-IDF vs BERT (%)", data=solapamiento_textuales, color="purple")
plt.title('Solapamiento entre recomendaciones TF-IDF y BERT por perfil')
plt.xlabel('Solapamiento (%)')
plt.ylabel('Perfil de usuario')
plt.axvline(x=50, color='red', linestyle='--', alpha=0.7)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## **5. Análisis de términos relevantes**

Para entender mejor cómo BERT captura la semántica que TF-IDF puede pasar por alto, analicemos ejemplos de consultas y similitudes.

In [ ]:
# Seleccionamos un perfil para analizar a profundidad
perfil_analizar = "Usuario primerizo"
consulta = next(p["text_query"] for p in test_profiles if p["nombre"] == perfil_analizar)

# Obtenemos top 5 recomendaciones de cada modelo
top_tfidf = all_recommendations["Numérico + TF-IDF"][perfil_analizar].head(5)
top_bert = all_recommendations["Numérico + BERT"][perfil_analizar].head(5)

print(f"Consulta: '{consulta}'")
print("\nRazas recomendadas por TF-IDF:")
for i, row in top_tfidf.iterrows():
    print(f"- {row['breed']} (Score: {row['cbf_score']:.4f})")
    
print("\nRazas recomendadas por BERT:")
for i, row in top_bert.iterrows():
    print(f"- {row['breed']} (Score: {row['cbf_score']:.4f})")

In [ ]:
# Análisis de términos importantes para TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer
import re

# Extraemos términos de la consulta
consulta_terms = consulta.split()

# Función para encontrar términos similares en el texto
def find_similar_terms(text, query_terms):
    found_terms = []
    text_lower = text.lower()
    for term in query_terms:
        if term.lower() in text_lower:
            found_terms.append(term)
    return found_terms

# Analizamos las razas top recomendadas por cada modelo
print("Términos encontrados en razas TF-IDF:")
for i, row in top_tfidf.iterrows():
    breed = row['breed']
    text = df[df['breed'] == breed]['text_combined'].values[0]
    terms = find_similar_terms(text, consulta_terms)
    print(f"- {breed}: {', '.join(terms) if terms else 'Ninguno'}")
    
print("\nTérminos encontrados en razas BERT:")
for i, row in top_bert.iterrows():
    breed = row['breed']
    text = df[df['breed'] == breed]['text_combined'].values[0]
    terms = find_similar_terms(text, consulta_terms)
    print(f"- {breed}: {', '.join(terms) if terms else 'Ninguno'}")

## **6. Análisis de términos semánticamente similares**

BERT puede capturar relaciones semánticas entre palabras que no son exactamente iguales. Vamos a analizar esto.

In [ ]:
# Función para encontrar términos semánticamente similares a la consulta
from sentence_transformers import util
import torch

def find_semantic_matches(text, query, model, threshold=0.5):
    # Tokenizar el texto en oraciones
    sentences = [s.strip() for s in re.split(r'[.!?]\s+', text) if s.strip()]
    
    if not sentences:
        return []
    
    # Calcular embeddings
    query_embedding = model.encode([query], convert_to_tensor=True)
    sentence_embeddings = model.encode(sentences, convert_to_tensor=True)
    
    # Calcular similaridades
    cosine_scores = util.cos_sim(query_embedding, sentence_embeddings)[0]
    
    # Encontrar oraciones con alta similaridad
    matches = []
    for i, score in enumerate(cosine_scores):
        if score > threshold:
            matches.append((sentences[i], score.item()))
    
    # Ordenar por similaridad
    return sorted(matches, key=lambda x: x[1], reverse=True)

# Analizamos las razas top de BERT para ver qué aspectos semánticos capturó
for i, row in top_bert.head(3).iterrows():
    breed = row['breed']
    text = df[df['breed'] == breed]['text_combined'].values[0]
    
    print(f"\n=== Análisis semántico para {breed} ===")
    semantic_matches = find_semantic_matches(text, consulta, bert_model, threshold=0.4)
    
    if semantic_matches:
        print("Oraciones semánticamente similares a la consulta:")
        for sentence, score in semantic_matches[:3]:  # Top 3 oraciones más similares
            print(f"- Score {score:.4f}: {sentence}")
    else:
        print("No se encontraron oraciones semánticamente similares con la similitud mínima.")

## **7. Evaluación de calidad**

Ahora vamos a evaluar la calidad de las recomendaciones generadas por los diferentes modelos.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# 7.1 Simulamos las preferencias "verdaderas" de los usuarios
# Función para simular favoritos con cada modelo
def simulate_favorites(profile, df, modelo="tfidf", n=10):
    """Simula razas favoritas según perfil y modelo"""
    # Extraemos preferencias numéricas y texto
    user_profile = {k: profile[k] for k in feature_order}
    user_text = profile.get("text_query", "")
    
    # Generamos vector numérico
    user_vec = generate_user_vector(user_profile, scaler, feature_order)
    num_similarities = cosine_similarity(user_vec, X_num)[0]
    
    if user_text:
        if modelo == "tfidf":
            # Método TF-IDF
            text_vec = tfidf_model.transform([user_text])
            text_similarities = cosine_similarity(text_vec, X_tfidf)[0]
        else:  # bert
            # Método BERT
            query_embedding = vectorize_query_bert(user_text, bert_model)
            text_similarities = cosine_similarity(query_embedding, X_bert)[0]
            
        # Normalizar similitudes
        num_similarities = (num_similarities - np.min(num_similarities)) / (np.max(num_similarities) - np.min(num_similarities) + 1e-10)
        text_similarities = (text_similarities - np.min(text_similarities)) / (np.max(text_similarities) - np.min(text_similarities) + 1e-10)
        
        # Combinar con alpha=0.5
        combined_similarities = 0.5 * num_similarities + 0.5 * text_similarities
    else:
        combined_similarities = num_similarities
    
    # Obtener índices de las n razas más similares
    top_indices = np.argsort(combined_similarities)[::-1][:n]
    return df.iloc[top_indices]['breed'].tolist()

# Generamos razas favoritas para cada perfil y modelo
ground_truth = {}
for modelo_nombre, modelo_config in {
    "Numérico": "tfidf", 
    "Numérico + TF-IDF": "tfidf", 
    "Numérico + BERT": "bert"
}.items():
    ground_truth[modelo_nombre] = {}
    
    for profile in test_profiles:
        nombre = profile["nombre"]
        
        if modelo_nombre == "Numérico":
            # Para el modelo numérico, no usamos texto
            profile_no_text = {k: v for k, v in profile.items() if k != "text_query"}
            ground_truth[modelo_nombre][nombre] = simulate_favorites(
                profile_no_text, df, modelo=modelo_config, n=10)
        else:
            ground_truth[modelo_nombre][nombre] = simulate_favorites(
                profile, df, modelo=modelo_config, n=10)

# Mostramos algunos resultados de preferencias simuladas
for modelo in ground_truth.keys():
    print(f"\nPreferencias simuladas para {perfil_analizar} con modelo {modelo}:")
    print(", ".join(ground_truth[modelo][perfil_analizar][:5]))

In [ ]:
# 7.2 Evaluamos métricas de calidad
from evaluation.metrics import precision_at_k, recall_at_k, ndcg_at_k

# Preparamos dataframe para resultados
eval_results = []

# Para cada perfil y cada modelo, evaluamos métricas
for profile in test_profiles:
    nombre = profile["nombre"]
    
    for modelo_nombre in ["Numérico", "Numérico + TF-IDF", "Numérico + BERT"]:
        # Obtenemos la verdad simulada adecuada
        true_favorites = ground_truth[modelo_nombre][nombre]
        
        # Obtenemos recomendaciones del modelo
        recommended_breeds = all_recommendations[modelo_nombre][nombre]['breed'].tolist()
        
        # Calculamos métricas para K=5 y K=10
        for k in [5, 10]:
            eval_results.append({
                'perfil': nombre,
                'modelo': modelo_nombre,
                'k': k,
                'precision': precision_at_k(recommended_breeds, true_favorites, k),
                'recall': recall_at_k(recommended_breeds, true_favorites, k),
                'ndcg': ndcg_at_k(recommended_breeds, true_favorites, k)
            })

# Convertimos a DataFrame
eval_df = pd.DataFrame(eval_results)

# Calculamos promedios por modelo y k
metrics_pivot = pd.pivot_table(
    eval_df,
    values=["precision", "recall", "ndcg"],
    index=["modelo", "k"],
    aggfunc="mean"
).reset_index()

# Mostramos resultados
metrics_pivot

In [ ]:
# Visualizamos resultados comparativos
for metric in ["precision", "recall", "ndcg"]:
    plt.figure(figsize=(12, 6))
    
    # Filtramos datos para el gráfico
    plot_data = metrics_pivot.pivot(index="k", columns="modelo", values=metric)
    
    # Creamos un gráfico de barras agrupadas
    plot_data.plot(kind="bar", rot=0, colormap="viridis")
    
    plt.title(f'{metric.upper()}@k promedio por modelo')
    plt.xlabel('k (número de recomendaciones)')
    plt.ylabel(f'{metric}@k')
    plt.legend(title="Modelo")
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

## **8. Análisis de casos específicos**

Analicemos algunos casos donde BERT muestra ventajas significativas sobre TF-IDF.

In [ ]:
# Encontramos casos donde las recomendaciones de BERT son mejores que TF-IDF
perfil_cases = {}

for profile in test_profiles:
    nombre = profile["nombre"]
    bert_ndcg = eval_df[(eval_df['perfil'] == nombre) & 
                         (eval_df['modelo'] == 'Numérico + BERT') & 
                         (eval_df['k'] == 10)]['ndcg'].values[0]
    
    tfidf_ndcg = eval_df[(eval_df['perfil'] == nombre) & 
                          (eval_df['modelo'] == 'Numérico + TF-IDF') & 
                          (eval_df['k'] == 10)]['ndcg'].values[0]
    
    perfil_cases[nombre] = {
        'bert_ndcg': bert_ndcg,
        'tfidf_ndcg': tfidf_ndcg,
        'diferencia': bert_ndcg - tfidf_ndcg
    }

# Ordenamos por diferencia
casos_ordenados = sorted(perfil_cases.items(), key=lambda x: x[1]['diferencia'], reverse=True)

# Mostramos casos con mayor diferencia
print("Perfiles donde BERT muestra mayor ventaja sobre TF-IDF:")
for nombre, metricas in casos_ordenados[:3]:
    print(f"\n{nombre}")
    print(f"NDCG BERT: {metricas['bert_ndcg']:.4f}")
    print(f"NDCG TF-IDF: {metricas['tfidf_ndcg']:.4f}")
    print(f"Diferencia: {metricas['diferencia']:.4f}")
    
    # Mostramos la consulta de texto
    consulta = next(p["text_query"] for p in test_profiles if p["nombre"] == nombre)
    print(f"Consulta: '{consulta}'")

In [ ]:
# Analizamos el perfil donde BERT muestra mayor ventaja
mejor_perfil = casos_ordenados[0][0]
consulta = next(p["text_query"] for p in test_profiles if p["nombre"] == mejor_perfil)

# Obtenemos top razas recomendadas por cada modelo
bert_recs = all_recommendations["Numérico + BERT"][mejor_perfil]
tfidf_recs = all_recommendations["Numérico + TF-IDF"][mejor_perfil]

# Comparamos razas top recomendadas
print(f"Consulta '{consulta}' para {mejor_perfil}\n")
print("Top 5 razas con BERT:")
for i, (_, row) in enumerate(bert_recs.head(5).iterrows(), 1):
    print(f"{i}. {row['breed']} (Score: {row['cbf_score']:.4f})")

print("\nTop 5 razas con TF-IDF:")
for i, (_, row) in enumerate(tfidf_recs.head(5).iterrows(), 1):
    print(f"{i}. {row['breed']} (Score: {row['cbf_score']:.4f})")

# Analizamos contenido de una raza recomendada por BERT pero no por TF-IDF
bert_breeds = bert_recs['breed'].tolist()[:5]
tfidf_breeds = tfidf_recs['breed'].tolist()[:5]
bert_unique = [b for b in bert_breeds if b not in tfidf_breeds]

if bert_unique:
    analyze_breed = bert_unique[0]
    print(f"\nAnálisis de raza recomendada por BERT pero no TF-IDF: {analyze_breed}")
    breed_text = df[df['breed'] == analyze_breed]['text_combined'].values[0]
    
    # Encontramos oraciones semánticamente similares a la consulta
    semantic_matches = find_semantic_matches(breed_text, consulta, bert_model, threshold=0.4)
    
    print("\nOraciones semánticamente similares:")
    for sentence, score in semantic_matches[:3]:
        print(f"- Score {score:.4f}: {sentence}")

## **9. Conclusiones y optimizaciones**

### **9.1 Hallazgos principales**

1. **Comparación de enfoques textuales**:
   - TF-IDF funciona bien para coincidencias exactas de palabras clave.
   - BERT captura mejor el significado semántico y puede detectar relaciones entre términos que no son exactamente iguales.

2. **Patrones en las recomendaciones**:
   - BERT tiende a capturar mejor consultas complejas con múltiples conceptos.
   - TF-IDF suele tener mejor rendimiento cuando los términos exactos de la consulta están en las descripciones de las razas.

3. **Scores de recomendación**:
   - Es normal que los scores absolutos difieran entre TF-IDF y BERT debido a la naturaleza diferente de los espacios vectoriales.
   - Lo importante es el ranking relativo de las recomendaciones, no el valor absoluto de los scores.

4. **Diversidad de recomendaciones**:
   - BERT tiende a producir recomendaciones más diversas semánticamente.
   - TF-IDF puede ofrecer mayor precisión en consultas específicas con términos exactos.

### **9.2 Optimizaciones futuras**

1. **Hibridación avanzada**:
   - Implementar un enfoque que combine TF-IDF y BERT basándose en la naturaleza de la consulta.
   - Explorar métodos de ensemble para combinar predicciones de múltiples modelos.

2. **Fine-tuning de BERT**:
   - Explorar el fine-tuning de BERT específicamente para el dominio de razas de perros.
   - Crear un conjunto de datos de pares (consulta, descripción) relevantes para este dominio.

3. **Expansión de consultas**:
   - Implementar expansión automática de consultas para enriquecer la entrada del usuario.
   - Utilizar grafos de conocimiento o relaciones semánticas para expandir términos relacionados con perros.

4. **Personalización adaptativa**:
   - Ajustar dinámicamente alpha basándose en la naturaleza de la consulta.
   - Desarrollar un modelo que aprenda qué enfoque (numérico, TF-IDF, BERT) funciona mejor para cada tipo de perfil.

### **9.3 Consideraciones de implementación**

1. **Eficiencia computacional**:
   - BERT es más costoso computacionalmente que TF-IDF.
   - Considerar preprocesar y almacenar embeddings para uso en producción.

2. **Interpretabilidad**:
   - Las recomendaciones de TF-IDF son más fáciles de explicar ("se recomienda porque contiene las palabras X, Y, Z").
   - Para BERT, desarrollar métodos para explicar las recomendaciones basadas en relaciones semánticas.

3. **Escalabilidad**:
   - Evaluar la necesidad de optimizaciones como cuantización de modelos o búsqueda aproximada de vecinos más cercanos para casos de uso a gran escala.